In [ ]:
!pip install -q kaggle numpy scipy matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive подключен!")
print("📁 Датасет будет храниться в: /content/drive/MyDrive/respiratory_sound_dataset/")

In [ ]:
import os
import shutil
from google.colab import files

KAGGLE_JSON_DRIVE = '/content/drive/MyDrive/kaggle.json'
KAGGLE_JSON_LOCAL = os.path.expanduser('~/.kaggle/kaggle.json')

if os.path.exists(KAGGLE_JSON_DRIVE):
    print("✅ kaggle.json найден в Google Drive!")
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(KAGGLE_JSON_DRIVE, KAGGLE_JSON_LOCAL)
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    print("✅ Kaggle API настроен из Drive!")
else:
    print("📤 kaggle.json не найден в Drive. Загрузите его:")
    uploaded = files.upload()
    
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    
    with open(KAGGLE_JSON_LOCAL, 'wb') as f:
        f.write(list(uploaded.values())[0])
    
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    
    shutil.copy(KAGGLE_JSON_LOCAL, KAGGLE_JSON_DRIVE)
    print("✅ Kaggle API настроен и сохранен в Drive!")

In [ ]:
import os
import shutil

DRIVE_DATASET_PATH = '/content/drive/MyDrive/respiratory_sound_dataset'
LOCAL_DATASET_PATH = '/content/respiratory_sound_dataset'

if os.path.exists(DRIVE_DATASET_PATH) and os.listdir(DRIVE_DATASET_PATH):
    print("✅ Датасет найден в Google Drive! Копирую локально...")
    if os.path.exists(LOCAL_DATASET_PATH):
        shutil.rmtree(LOCAL_DATASET_PATH)
    shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
    print("✅ Датасет скопирован из Drive!")
else:
    print("📥 Датасет не найден в Drive. Загружаю с Kaggle (3.69GB, ~5-10 мин)...")
    !kaggle datasets download -d vbookshelf/respiratory-sound-database
    
    print("📦 Распаковка...")
    os.makedirs(LOCAL_DATASET_PATH, exist_ok=True)
    !unzip -q respiratory-sound-database.zip -d {LOCAL_DATASET_PATH}
    !rm respiratory-sound-database.zip
    
    print("💾 Сохраняю в Google Drive для следующих запусков...")
    os.makedirs(os.path.dirname(DRIVE_DATASET_PATH), exist_ok=True)
    if os.path.exists(DRIVE_DATASET_PATH):
        shutil.rmtree(DRIVE_DATASET_PATH)
    shutil.copytree(LOCAL_DATASET_PATH, DRIVE_DATASET_PATH)
    print("✅ Датасет сохранен в Drive!")

print(f"\n📊 Датасет готов к использованию: {LOCAL_DATASET_PATH}")
!ls -lh {LOCAL_DATASET_PATH} | head -20

In [ ]:
!git clone https://github.com/incRED1bl/course_paper.git
%cd course_paper

In [ ]:
import os

print("📁 Проверка структуры датасета:")
print("\nСодержимое /content/respiratory_sound_dataset:")
dataset_path = '/content/respiratory_sound_dataset'

if os.path.exists(dataset_path):
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # показываем первые 5 файлов
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... и еще {len(files) - 5} файлов')
        if level > 2:  # ограничиваем глубину
            break
else:
    print(f"❌ Путь {dataset_path} не существует!")

print("\n🔍 Поиск .wav файлов:")
for root, dirs, files in os.walk(dataset_path):
    wav_files = [f for f in files if f.endswith('.wav')]
    if wav_files:
        print(f"Найдено {len(wav_files)} .wav файлов в: {root}")
        print(f"Примеры: {wav_files[:3]}")
        break

In [ ]:
import os
import numpy as np
from scipy import signal
from scipy.io import wavfile

def load_respiratory_sounds(dataset_path, max_files=20):
    """Load respiratory sound data from the dataset."""
    signals = {}
    
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset path not found: {dataset_path}")
    
    # Поиск папки audio_and_txt_files в любом месте датасета
    audio_dir = None
    for root, dirs, files in os.walk(dataset_path):
        if 'audio_and_txt_files' in dirs:
            audio_dir = os.path.join(root, 'audio_and_txt_files')
            break
    
    if audio_dir and os.path.exists(audio_dir):
        print(f"📁 Найдена папка с аудио: {audio_dir}")
        wav_files = [f for f in os.listdir(audio_dir) if f.endswith('.wav')]
        print(f"📊 Всего .wav файлов: {len(wav_files)}, загружаем: {min(max_files, len(wav_files))}")
        
        for wav_file in wav_files[:max_files]:
            file_path = os.path.join(audio_dir, wav_file)
            try:
                sample_rate, signal_data = wavfile.read(file_path)
                signals[wav_file] = {
                    'signal': signal_data,
                    'sample_rate': sample_rate
                }
            except Exception as e:
                print(f"Warning: Could not load {wav_file}: {e}")
    else:
        print(f"❌ Папка audio_and_txt_files не найдена в {dataset_path}")
    
    return signals


def extract_frequency_features(audio_data, sample_rate, n_fft=2048):
    """Extract frequency domain features."""
    fft_result = np.fft.rfft(audio_data, n=n_fft)
    frequencies = np.fft.rfftfreq(n_fft, 1/sample_rate)
    magnitudes = np.abs(fft_result)
    
    feature_vector = {
        'spectral_centroid': np.sum(frequencies * magnitudes) / np.sum(magnitudes),
        'spectral_bandwidth': np.sqrt(np.sum(((frequencies - np.sum(frequencies * magnitudes) / np.sum(magnitudes)) ** 2) * magnitudes) / np.sum(magnitudes)),
        'spectral_rolloff': frequencies[np.where(np.cumsum(magnitudes) >= 0.85 * np.sum(magnitudes))[0][0]],
        'low_freq_energy': np.sum(magnitudes[frequencies < 500]),
        'mid_freq_energy': np.sum(magnitudes[(frequencies >= 500) & (frequencies < 1000)]),
        'high_freq_energy': np.sum(magnitudes[frequencies >= 1000]),
        'peak_frequency': frequencies[np.argmax(magnitudes)]
    }
    
    return frequencies, magnitudes, feature_vector


def detect_whistles(audio_data, sample_rate, whistle_freq_range=(400, 1600)):
    """Detect whistle sounds in audio."""
    f, t, Sxx = signal.spectrogram(audio_data, sample_rate, nperseg=256)
    
    freq_mask = (f >= whistle_freq_range[0]) & (f <= whistle_freq_range[1])
    whistle_energy = np.sum(Sxx[freq_mask, :])
    total_energy = np.sum(Sxx)
    
    whistle_strength = whistle_energy / total_energy if total_energy > 0 else 0
    whistle_detected = whistle_strength > 0.2
    
    return whistle_detected, whistle_strength

print("✅ Функции загрузки данных готовы!")

In [ ]:
import sys
import matplotlib.pyplot as plt

from app.entropy_complexity import compute_entropy_complexity
from app.feature_extraction import extract_all_features, build_feature_summary, print_feature_summary

print("✅ Модули анализа загружены!")

In [ ]:
def plot_time_frequency_spectra(signals, features_dict, sample_rate):
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    fig.suptitle('Frequency Spectrum Analysis of Lung Sounds', fontsize=16, fontweight='bold')
    
    colors_map = {'Healthy': 'green', 'Diseased': 'red', 'COPD': 'orange'}
    
    for idx, (disease_name, signal_data) in enumerate(signals.items()):
        frequencies = features_dict[disease_name]['frequencies']
        magnitudes = features_dict[disease_name]['magnitudes']
        color = colors_map.get(disease_name, 'blue')
        
        axes[idx, 0].plot(signal_data[:1000], linewidth=0.8, color=color)
        axes[idx, 0].set_title(f'{disease_name} - Time Domain')
        axes[idx, 0].set_xlabel('Samples')
        axes[idx, 0].set_ylabel('Amplitude')
        axes[idx, 0].grid(True, alpha=0.3)
        
        axes[idx, 1].plot(frequencies, magnitudes, linewidth=1.2, color=color)
        axes[idx, 1].axvspan(400, 1600, alpha=0.2, color='red', label='Whistle Range')
        axes[idx, 1].axvspan(0, 500, alpha=0.1, color='green', label='Normal Range')
        axes[idx, 1].set_title(f'{disease_name} - Frequency Spectrum')
        axes[idx, 1].set_xlabel('Frequency (Hz)')
        axes[idx, 1].set_ylabel('Magnitude')
        axes[idx, 1].set_xlim(0, 2000)
        axes[idx, 1].grid(True, alpha=0.3)
        axes[idx, 1].legend(loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()


def plot_energy_distribution(disease_features):
    diseases = list(disease_features.keys())
    colors = ['#2ecc71', '#e74c3c', '#e67e22']
    
    energy_features = ['Low Freq Energy', 'Mid Freq Energy', 'High Freq Energy']
    energy_data = np.array([[disease_features[d][f] for f in energy_features] for d in diseases])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(diseases))
    width = 0.25
    for i, feature in enumerate(energy_features):
        ax.bar(x + i*width, energy_data[:, i], width, label=feature, alpha=0.8)
    
    ax.set_title('Energy Distribution by Frequency Band', fontsize=14, fontweight='bold')
    ax.set_xlabel('Disease Category')
    ax.set_ylabel('Relative Energy')
    ax.set_xticks(x + width)
    ax.set_xticklabels(diseases)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()


def plot_whistle_strength(disease_features):
    diseases = list(disease_features.keys())
    colors = ['#2ecc71', '#e74c3c', '#e67e22']
    whistle_strengths = [disease_features[d]['Whistle Strength'] for d in diseases]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(diseases, whistle_strengths, color=colors, alpha=0.8)
    ax.set_title('Whistle Strength by Disease Category', fontsize=14, fontweight='bold')
    ax.set_ylabel('Whistle Strength')
    ax.set_ylim(0, 1)
    ax.axhline(y=0.2, color='r', linestyle='--', linewidth=2, label='Detection Threshold')
    
    for i, (disease, strength) in enumerate(zip(diseases, whistle_strengths)):
        ax.text(i, strength + 0.03, f'{strength:.3f}', ha='center', fontweight='bold')
    
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()


def plot_peak_frequencies(disease_features):
    diseases = list(disease_features.keys())
    colors = ['#2ecc71', '#e74c3c', '#e67e22']
    peak_freqs = [disease_features[d]['Peak Frequency'] for d in diseases]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(diseases, peak_freqs, color=colors, alpha=0.8)
    ax.set_title('Peak Frequency by Disease Category', fontsize=14, fontweight='bold')
    ax.set_ylabel('Frequency (Hz)')
    ax.axhspan(400, 1600, alpha=0.2, color='red', label='Whistle Range (400-1600 Hz)')
    ax.axhspan(0, 300, alpha=0.2, color='green', label='Normal Range (0-300 Hz)')
    
    for i, (disease, freq) in enumerate(zip(diseases, peak_freqs)):
        ax.text(i, freq + 20, f'{freq:.1f} Hz', ha='center', fontweight='bold')
    
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()


def plot_entropy_complexity_plane(disease_features):
    diseases = list(disease_features.keys())
    colors = ['#2ecc71', '#e74c3c', '#e67e22']
    entropies = [disease_features[d]['Entropy'] for d in diseases]
    complexities = [disease_features[d]['Complexity'] for d in diseases]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(entropies, complexities, s=300, c=colors, alpha=0.6, edgecolors='black', linewidths=2)
    
    for i, disease in enumerate(diseases):
        ax.annotate(disease, (entropies[i], complexities[i]), 
                    xytext=(10, 10), textcoords='offset points', 
                    fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.5', facecolor=colors[i], alpha=0.3))
    
    ax.set_title('Entropy-Complexity Plane for Disease Classification', fontsize=14, fontweight='bold')
    ax.set_xlabel('Normalized Entropy (H)', fontsize=12)
    ax.set_ylabel('Statistical Complexity (C)', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def plot_whistle_vs_centroid(disease_features):
    diseases = list(disease_features.keys())
    colors = ['#2ecc71', '#e74c3c', '#e67e22']
    whistle_strengths = [disease_features[d]['Whistle Strength'] for d in diseases]
    centroids = [disease_features[d]['Spectral Centroid'] for d in diseases]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(whistle_strengths, centroids, s=300, c=colors, alpha=0.6, edgecolors='black', linewidths=2)
    
    for i, disease in enumerate(diseases):
        ax.annotate(disease, (whistle_strengths[i], centroids[i]), 
                    xytext=(10, 10), textcoords='offset points', 
                    fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.5', facecolor=colors[i], alpha=0.3))
    
    ax.set_title('Whistle Strength vs Spectral Centroid', fontsize=14, fontweight='bold')
    ax.set_xlabel('Whistle Strength', fontsize=12)
    ax.set_ylabel('Spectral Centroid (Hz)', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("✅ Функции визуализации загружены!")

In [ ]:
dataset_path = '/content/respiratory_sound_dataset'

print("📂 Загрузка аудио файлов...")
# Измените max_files для загрузки большего количества файлов
# max_files=20 - для быстрого тестирования
# max_files=100 - для среднего размера выборки  
# max_files=920 - все файлы (может занять время!)
signals = load_respiratory_sounds(dataset_path, max_files=20)

print(f"✅ Загружено {len(signals)} файлов")
print(f"📝 Примеры: {list(signals.keys())[:3]}")

In [ ]:
signal_data = {name: data['signal'] for name, data in signals.items()}
sample_rate = list(signals.values())[0]['sample_rate']

print(f"Sample rate: {sample_rate} Hz")

print("\n🔬 Извлечение признаков...")
features_dict, whistle_results, entropy_complexity = extract_all_features(
    signal_data, sample_rate, m=3, tau=1
)

disease_features = build_feature_summary(
    signal_data, features_dict, whistle_results, entropy_complexity
)

print("✅ Признаки извлечены!")
print(f"\n📊 Количество обработанных файлов: {len(disease_features)}")

In [ ]:
sample_signals = {k: signal_data[k] for k in list(signal_data.keys())[:3]}
sample_features = {k: features_dict[k] for k in list(features_dict.keys())[:3]}
sample_disease_features = {k: disease_features[k] for k in list(disease_features.keys())[:3]}

plot_time_frequency_spectra(sample_signals, sample_features, sample_rate)

In [ ]:
plot_energy_distribution(sample_disease_features)

In [ ]:
plot_entropy_complexity_plane(sample_disease_features)

In [ ]:
print("💡 Добавьте здесь код обучения вашей модели")

In [ ]:
import pickle

with open('features.pkl', 'wb') as f:
    pickle.dump({
        'features_dict': features_dict,
        'whistle_results': whistle_results,
        'entropy_complexity': entropy_complexity,
        'disease_features': disease_features
    }, f)

print("✅ Признаки сохранены в features.pkl")

from google.colab import files
files.download('features.pkl')

In [ ]:
print("="*70)
print("  ИТОГОВЫЕ РЕЗУЛЬТАТЫ")
print("="*70)
print(f"\n📊 Обработано файлов: {len(signals)}")
print(f"🔬 Извлечено признаков: {len(disease_features)}")
print(f"\n✅ Анализ завершен!")
print(f"\n💾 Не забудьте скачать:")
print(f"   - features.pkl (признаки)")
print(f"   - model.pkl (обученная модель)")
print(f"\n🚀 Загрузите файлы в VS Code для дальнейшей работы")